In [ ]:
# read in data
# germinal(processed): Name,Well in plate,Target,Protein_sequence,KD[M],KD1[M],KD2[M]
# nb_scaffold: Nanobody,Target,Sequence,KD[M],Notes
# target: Target,Target_Residues,Target_Sequence
# create two separate dfs for germinal and nb_scaffold (save them)
# then create a merged df and merge with target df on 'Target' separately
# generate fasta files by concatenating protein and target sequences separated by ":"

# read in the two original separate dfs for target shuffle
# shuffle target
# merge with target on target
# generate fasta file with shuffled targets

#### prepare germinal ds

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# read excel with multiple sheets
excel_file = "/home/bri/pymol/datasets/germinal/germinal_examples.xlsx" 
dfs = pd.read_excel(excel_file, sheet_name=None)

# combine all sheets into one df
germinal_df = pd.concat(dfs.values(), ignore_index=True)
# germinal_df.columns = germinal_df.columns.str.lower()

# save
germinal_df.to_csv("/home/bri/pymol/datasets/germinal/germinal.csv", index=False)
print("Saved combined data from all sheets to germinal.csv")

Saved combined data from all sheets to germinal.csv


In [24]:
# Read in data
import pandas as pd

targets_df = pd.read_csv("/home/bri/pymol/datasets/target/targets.txt")
#germinal_df = pd.read_csv("/home/bri/pymol/datasets/germinal/germinal_target_swap.csv")
nb_scaffold_df = pd.read_csv("/home/bri/pymol/datasets/nb_scaffold/nb_scaffold_target_swap.csv")


# drop column label
#germinal_df.drop(columns=['label'], inplace=True)
# merge on 'target'
#merged_germinal_df = pd.merge(germinal_df, targets_df, on='target')
merged_nb_scaffold_df = pd.merge(nb_scaffold_df, targets_df, on='target', how='left')

# add a column "type" and set it to real
#merged_germinal_df['type'] = 'target_shuffle'
merged_nb_scaffold_df['type'] = 'real'

# modify "name" to "nanobody" and "protein_sequence" to "nanobody_sequence" in both dfs
#merged_germinal_df.rename(columns={'name': 'id', 'protein_sequence': 'vhh_sequence'}, inplace=True)
merged_nb_scaffold_df.rename(columns={'name': 'id', 'protein_sequence': 'vhh_sequence'}, inplace=True)

# add a colunm "binder" which is 1 if theres a KD or EC50 value and 0 if not
#merged_germinal_df['binder'] = 0 #merged_germinal_df['KD[M]'].apply(lambda x: 1 if pd.notnull(x) else 0)
merged_nb_scaffold_df['binder'] = merged_nb_scaffold_df['KD[M]'].apply(lambda x: 1 if pd.notnull(x) else 0) | merged_nb_scaffold_df['EC50[M]'].apply(lambda x: 1 if pd.notnull(x) else 0)

# add a column "source" to indicate which df the data came from
#merged_germinal_df['source'] = 'germinal'
merged_nb_scaffold_df['source'] = 'mcmahon'

# rearrange columns
merged_nb_scaffold_df = merged_nb_scaffold_df[['source', 'type','id','vhh_sequence', 'target','target_residues', 'target_sequence','binder','KD[M]', 'EC50[M]','notes', 'comments(target)']]
#merged_germinal_df = merged_germinal_df[['source', 'type', 'id', 'well', 'vhh_sequence', 'target','target_residues', 'target_sequence','binder','KD[M]', 'KD1[M]', 'KD2[M]', 'comments(target)']]
#merged_germinal_df.drop(columns=['well', 'KD1[M]', 'KD2[M]'], inplace=True)


# Inspect merged data
merged_nb_scaffold_df.tail()
#merged_germinal_df.tail()


# Save combined data to a single CSV
# reorder columns to put comments(target) at the end

#merged_germinal_df.to_csv("/home/bri/pymol/datasets/germinal/germinal_swap.csv", index=False)
merged_nb_scaffold_df.to_csv("/home/bri/pymol/datasets/nb_scaffold/nb_scaffold_shuffled_targets.csv", index=False)
print("Saved combined data from all sheets to germinal.csv and nb_scaffold.csv")

Saved combined data from all sheets to germinal.csv and nb_scaffold.csv


In [8]:
# join the two dfs together 
combined_df = pd.concat([merged_germinal_df, merged_nb_scaffold_df], ignore_index=True)
combined_df = combined_df[['source', 'type', 'id', 'vhh_sequence', 'target', 'target_residues', 'target_sequence', 'binder', 'KD[M]', 'EC50[M]','comments(target)', 'notes']]

# save combined df as a csv
combined_df.to_csv("/home/bri/pymol/datasets/combined_targets.csv", index=False)

# save as excel
combined_df.to_excel("/home/bri/pymol/datasets/combined_targets.xlsx", index=False)

In [25]:
binders = combined_df[combined_df['binder'] == 1]
print(f"Number of binders: {len(binders)}")

Number of binders: 30


In [ ]:
def generate_fasta(df, filename):
    with open(filename, 'w') as f:
        for _, row in df.iterrows():
            header = f">{row['name']}+{row['target']}({row['target_residues']})"
            seq = f"{row['protein_sequence']}:{row['target_sequence']}"
            f.write(f"{header}\n{seq}\n")
    print(f"FASTA file generated: {filename}")

# Generate FASTA files
generate_fasta(merged_germinal_df, 'germinal_targets.fasta')
generate_fasta(merged_nb_scaffold_df, 'nb_scaffold_targets.fasta')

FASTA file generated: germinal_targets.fasta
FASTA file generated: nb_scaffold_targets.fasta


In [ ]:
# TO DO
# add identifying columns (binder, source, original/shuffled)
# rearrange columns
# create table for the 3 Ag ones, add mutants
# template?


merged_germinal_df missing columns: ['name', 'protein_sequence']
merged_germinal_df reordered to: ['well', 'target', 'target_residues', 'target_sequence', 'KD[M]', 'KD1[M]', 'KD2[M]']
merged_nb_scaffold_df missing columns: ['name', 'well', 'protein_sequence', 'KD1[M]', 'KD2[M]']
merged_nb_scaffold_df reordered to: ['target', 'target_residues', 'target_sequence', 'KD[M]']
Reordered dataframes saved.


#### antigen-swap negatives
: take known antibody–antigen binders and swap the antigen partner to create realistic non-binding decoys.

In [ ]:
import pandas as pd
import numpy as np

germinal_df = pd.read_csv("nb_scaffold_paper.csv")
df_neg = germinal_df.copy()
df_neg["target"] = np.random.permutation(df_neg["target"])

# remove accidental true pairs
df_neg = df_neg[df_neg["target"] != germinal_df["target"]]

df_neg["label"] = 0

df_neg.head(18)
df_neg.value_counts("target")

df_neg.to_csv("germinal_target_shuffle.csv", index=False)



target
B2AR           6
Adiponectin    4
A2AR           2
HSA            1
Name: count, dtype: int64